# **Objective**
In this notebook, I demonstrate that it is possible to find a compensating charge $z(\mathbf{r})$ for the electron charge $n(\mathbf{r})$, such that the Hartree potential of the sum of the two calculated with Ewald's method, $V_{H,\mathbf{G} \neq 0}[n+z](\mathbf{r})$, is 0 when $\mathbf{r} \notin U_a$, where $U_a$ is the region where $n$ and $z$ is localized.

To achieve the above, it suffices to make all the $l =2$ cartesian moments as well as all the spherical multipole moment of $n+z$ zero.

Note that making all $l=2$ cartesian moments zero necessarily makes all $l=2$ spherical moments zero. In fact,

$$
l=2\text{ cartesian moments zero}\\
⇔\\
l=2\text{ spherical moments zero} ∧ \int d\mathbf{r}\, r^2 [n+z](\mathbf{r}) = 0
$$

Therefore, we see that making $l=2$ cartesian moments zero induces some extra $l=0$ term, which then needs to be accounted for in the compensating charge $z$.

## **Constructing $n$**
Let's see start from the simplest case, where the electron charge is s-type
$$
n(\mathbf{r}) = N_1 e^{-\alpha_1 r^2}
$$
where $N_1$ is the normalization constant such that $q = \int d\mathbf{r}\, n(\mathbf{r})$. Explicitly,
$$
N_1 = q \left(\frac{\alpha_1}{\pi} \right) ^ {3/2}
$$

In [166]:
from pyscf.pbc import gto as pgto
import pyscf

import numpy as np

# cell params
L = 10
x = L/2
ke_cutoff = 200

# electron charge
alpha1 = 9

basis = {'He': [[0, [alpha1, 1.]]]}

mol = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = basis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L
)

# evaluate GTO vals on fft grid
mesh = pyscf.pbc.tools.cutoff_to_mesh(mol.lattice_vectors(), mol.ke_cutoff)
Rgrid = mol.get_uniform_grids(mesh=mesh, wrap_around=False)
aoOnR = mol.pbc_eval_gto('GTOval', Rgrid)
dv = mol.vol/Rgrid.shape[0]

# normalize properly
q = 1
N1 = q * (alpha1/np.pi)**(3/2) * (np.pi/alpha1/2)**(3/4)
nOnR = N1 * aoOnR[:, 0]

Test that $n$ as constructed is properly normalized

In [167]:
1 - np.sum(nOnR) * dv

np.float64(1.4553680482976006e-10)

### **Calculate the Ewald potential**

In [168]:
def get_Gv(nmesh,reciprocal_vecs):
    rx = np.fft.fftfreq(nmesh[0], 1./nmesh[0])
    ry = np.fft.fftfreq(nmesh[1], 1./nmesh[1])
    rz = np.fft.fftfreq(nmesh[2], 1./nmesh[2])
    return np.dot(pyscf.lib.cartesian_prod((rx,ry,rz)), reciprocal_vecs).astype(np.float64)

def getFormFactor(nmesh,cell):
    G2 = get_Gv(nmesh,cell.reciprocal_vectors())**2
    G2 = np.sum( G2, axis = 1)
    FF = np.zeros((G2.shape[0]), np.float64)
    idx = np.greater( G2, 0.)
    FF[idx] = 4. * np.pi /G2[idx]
    return FF

In [169]:
FF = getFormFactor(mesh, mol).reshape(mesh)
vnOnR = np.fft.ifftn(np.fft.fftn(nOnR.reshape(mesh)) * FF).real.flatten()

## **Constructing $z$**
We now construct the compensating charge $z$:
$$
z(\mathbf{r}) = N_2 e^{-\alpha_2 r^2}
$$
where $N_2$ is the normalization constant such that $\int d\mathbf{r} \, z(\mathbf{r}) = -q$

In [170]:
# spherical compensating charge
alpha2 = 4.

gbasis = {'He': [[0, [alpha2, 1.]]]}

gmol = pgto.M(
    atom = mol.atom,
    basis = gbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L
)

# evaluate GTO vals on fft grid
gOnR = gmol.pbc_eval_gto('GTOval', Rgrid)

# normalize properly
q = 1
N2 = q * (alpha2/np.pi)**(3/2) * (np.pi/alpha2/2)**(3/4)
zOnR = N2 * gOnR[:, 0]

Check that $z$ is normalized

In [171]:
1 - zOnR.sum() * dv

np.float64(5.882216935759743e-11)

In [172]:
vzOnR = np.fft.ifftn(np.fft.fftn(zOnR.reshape(mesh)) * FF).real.flatten()

## **The Problem**
Now, as constructed, you might expect $V_{H, \mathbf{G} \neq 0} [n + z] (\mathbf{r}) = 0$ when $\mathbf{r} \notin U_a$, because all (spherical) multipole moment of $n+z$ vanishes (meaning that $V_{H, exact}(\mathbf{r}) = 0$ when $r \notin U_a$). However, this is NOT the case as shown below

In [173]:
# determine interstitial region
eps = 1.e-5
mask = zOnR < eps # mask for points outside of Ua

# the total hartree potential
vnzOnR = vnOnR - vzOnR

print(vnzOnR[mask].max())
print(vnzOnR[mask].min())

-6.408158225051919e-05
-6.466275442312819e-05


One can see from above that the potential far away is a non-zero constant. In fact, this constant is given by
$$
V_{H, \mathbf{G} \neq 0}[n+z] = V_{H, exact}[n+z] - V_{H, \mathbf{G} = 0}[n+z]
$$
where
$$
V_{H, \mathbf{G} = 0}[n+z](\mathbf{r}) = -\frac{4 \pi}{\Omega}\frac{1}{6}\int_{cell} d\mathbf{r} \, r^2 [n(\mathbf{r}) + z(\mathbf{r})]
$$
This does not have a simple analytical solution. In fact, the best I can do is to write:
$$
\int_{cell} d\mathbf{r} \, r^2 n(\mathbf{r}) = N_1 \sum_\mathbf{R} \int_{cell - \mathbf{R}} d\mathbf{r} \, (\mathbf{r} + \mathbf{R})^2 e^{-\alpha_1 (\mathbf{r} - \mathbf{r}_1)^2}
$$
So let's calculate this numerically:

In [174]:
def vG0(dens, Rgrid, dv, vol):
    assert(dens.shape[0] == Rgrid.shape[0])
    r2 = np.sum(Rgrid**2, axis=1)

    return -(4*np.pi/vol)*(1/6) * np.dot(dens, r2) * dv

Indeed, at the box edge (well outside $U_a$)
$$
V_{H, \mathbf{G} \neq 0}[n+z](\mathbf{r}) + V_{H, \mathbf{G} = 0}[n+z](\mathbf{r}) = 0 = V_{H, exact}[n+z](\mathbf{r})
$$

In [175]:
(vnOnR[0] - vzOnR[0]) + vG0(nOnR-zOnR, Rgrid, dv, mol.vol)

np.float64(1.0878684292368185e-11)

## **The Solution**
As illustrated above, simply making all the (spherical) multipole moment zero is not enough to make $V_{H, \mathbf{G} = 0} (\mathbf{r}) = 0$ when $\mathbf{r}  \notin U_a$. What we need to do instead is the following:
1. Make all the cartesian $l=2$ moments of $[n+z](\mathbf{r})$ zero.
1. Make sure all spherical moments remain zero. This would result in a different $l=0$ component for $z$.

In this particular case, it suffices to consider $z(\mathbf{r})$ of the form:
$$
z(\mathbf{r}) = \left(C_0 + C_{2,x^2}x^2 + C_{2,y^2}y^2 + C_{2,z^2}z^2\right) e^{-\alpha_2 r^2}
$$
and solve the following set of equations:
$$
\begin{cases}
\int d\mathbf{r} \, x^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, y^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, z^2 [n+z](\mathbf{r}) = 0\\
\int d\mathbf{r} \, [n+z](\mathbf{r}) = 0
\end{cases}
$$
The first 3 equations allow us to write $C_{2, x^2} = C_{2, y^2} = C_{2, z^2} \equiv C_2$, that is
$$
z(\mathbf{r}) = \left(C_0 + C_{2}r^2 \right) e^{-\alpha_2 r^2}
$$
where $C_2$ can be written in terms of $C_0$. The last equation then allows us to solve for $C_0$ and $C_2$:
$$
\begin{cases}
C_0 = N_2 \left( \frac{3\alpha_2}{2\alpha_1} - \frac{5}{2}\right)\\
C_2 = N_2 \left( \alpha_2 - \frac{\alpha_2^2}{\alpha_1} \right)
\end{cases}
$$
Note that the above reduce to the trivial case when $\alpha_1 = \alpha_2$

In [176]:
# cartesian compensating charge
alpha2 = 4
cbasis = {'He': [[0, [alpha2, 1.]], [2, [alpha2, 1.]]]}

cmol = pgto.M(
    atom = mol.atom,
    basis = cbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L,
    cart = True,
)

smol = pgto.M(
    atom = mol.atom,
    basis = cbasis,
    ke_cutoff=mol.ke_cutoff,
    a = np.eye(3)*L,
    cart = False,
)

# evaluate GTO vals on fft grid
cOnR = cmol.pbc_eval_gto('GTOval', Rgrid)
sOnR = smol.pbc_eval_gto('GTOval', Rgrid)

from scipy.special import gamma
# normalize properly
def normalize_cart_gto(ao, l, alpha):
    # normalize primitive cartesian gto
    n = l + 1.5
    return ao * (gamma(n)/(2 * (2*alpha)**n))**(1/2)

cOnR_0 = cOnR[:, 0] * (np.pi/alpha2/2)**(3/4)
cOnR_0_test = normalize_cart_gto(cOnR[:, 0], 0, alpha2)
cOnR_xx = normalize_cart_gto(cOnR[:, 1], 2, alpha2)
cOnR_yy = normalize_cart_gto(cOnR[:, 4], 2, alpha2)
cOnR_zz = normalize_cart_gto(cOnR[:, 6], 2, alpha2)

NN2 = q * (alpha2/np.pi)**(3/2)

C0 = NN2 * (3*alpha2/2/alpha1 - 5/2)
C2 = NN2 * (alpha2 - alpha2**2/alpha1)
zOnR_cart = C0 * cOnR_0 + C2 * (cOnR_xx + cOnR_yy + cOnR_zz)

vcOnR = np.fft.ifftn(np.fft.fftn(zOnR_cart.reshape(mesh)) * FF).real.flatten()

In [177]:
print(C0, C2)

-2.633944457835777 3.192659948891851


In [178]:
print(sOnR[:, 0].sum())
print(cOnR[:, 0].sum())

386.9088297527389
386.9088297527389


In [179]:
print(cOnR_0.sum() * dv)
print(cOnR_0_test.sum() * dv)

0.6960409995630208
0.1963495408378124


In [180]:
(np.pi/4)**(3/2)

0.6960409996039635

In [181]:
# determine interstitial region
eps = 1.e-5
masks = cOnR < eps # mask for points outside of Ua
mask = np.logical_and.reduce(masks.T, axis=0)

vncOnR = vnOnR + vcOnR

print(vncOnR[mask].max())
print(vncOnR[mask].min())

4.274352150002159e-09
-9.774224490888272e-08


One can see that the potential $V_{H, \mathbf{G} \neq 0 } [n + z] (\mathbf{r}) = 0$ when $\mathbf{r} \notin U_a$, when $z(\mathbf{r})$ is constructed as above.

# **The General Case**
The general strategy is to implement this solution is the following:
1. For $l=0$ and $l=2$, use the cartesian GTO as compensating charge to fit the pair density $\phi_P \phi_Q$
1. For the rest of the angular momentum, use spherical GTO as compensating charge (as before).

First, let us consider the $l=0, x^2, y^2, z^2$ subspace of the cartesian compensating charge. This is the only non-trivial subspace that requires care because they are coupled with each other, as shown in the previous example. The compensating charge has the general form as follows:
$$
z(\mathbf{r}) = \sum_{g \in \{ S_{00}, x^2, y^2, z^2\}} M_{PQ, g} N_g g e^{-\alpha r^2}
$$
where
$$
N_g = \left( \frac{\Gamma(l_g+1.5)}{2 (2\alpha_g)^{l_g+1.5}} \right) ^ {-\frac{1}{2}}
$$
which is just the basis_norm already implemented.

The system of equation that needs to be solved is:
$$
\begin{cases}
\frac{1}{6} N_P N_Q\bar{\Gamma}(l^{(2)}_{PQ}, \alpha_{PQ}) \left(2+ 3 \mathcal{N}_{22} \mathcal{C}^{22}_{L_PL_Q} - \mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot x^2}\\
\frac{1}{6} N_P N_Q \bar{\Gamma}(l^{(2)}_{PQ}, \alpha_{PQ}) \left(2- 3 \mathcal{N}_{22} \mathcal{C}^{22}_{L_PL_Q} - \mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot y^2}\\
\frac{1}{3} N_P N_Q \bar{\Gamma}(l^{(2)}_{PQ}, \alpha_{PQ}) \left(1+\mathcal{N}_{20} \mathcal{C}^{20}_{L_P L_Q}\right) = \sum_g M_{PQ, g} N_g I_{g \cdot z^2}\\
N_P N_Q \mathcal{N}_{00} \mathcal{C}^{00}_{L_P L_Q} \bar{\Gamma}(l^{(0)}_{PQ}, \alpha_{PQ}) = \sum_g M_{PQ,g} N_g I_g
\end{cases}
$$
where
$$
\bar{\Gamma}(l^{(n)}_{PQ}, \alpha_{PQ}) \equiv \int_0^\infty dr \, r^{l^{(n)}_{PQ}} e^{-\alpha_{PQ}r^2} = \frac{\Gamma \left((l^{(n)}_{PQ} + 1)/2\right)}{2\alpha_{PQ}^{(l^{(n)}_{PQ} + 1)/2}}\\
l^{(n)}_{PQ} \equiv n + 2 + l_P + l_Q, \alpha_{PQ} = \alpha_P + \alpha_Q
$$
and
$$
\mathcal{N_{00}} = \sqrt{4 \pi}\\
\mathcal{N_{20}} = 4\sqrt{\frac{\pi}{5}}\\
\mathcal{N_{22}} = 4\sqrt{\frac{\pi}{15}}\\
$$
and
$$
I_{f(\mathbf{r})} \equiv \int d\mathbf{r} \, f(\mathbf{r}) e^{-\alpha r^2}
$$
and $\mathcal{C}^{L_g}_{L_P L_Q}$ is the Clebsh-Gordan coefficients.

In [182]:
from pyscf.paw import ClebschGordan
import scipy

CG = ClebschGordan.RealCG

# define some constants
def gammaBar(n, l1, l2, alpha1, alpha2):
    L = n + 2 + l1 + l2
    LL = (L+1)/2
    a = alpha1 + alpha2
    return scipy.special.gamma(LL) / (2*a**(LL))

def I(alpha, x: int, y: int, z: int):
    # take powers of xyz and compute integral

    xyz = np.array([x, y, z])
    n = (xyz+1)/2
    Ixyz = scipy.special.gamma(n) / (alpha**n)

    return np.prod(Ixyz)

def basisnorm(alpha, l):
    L = l*2+2
    n = 0.5*(L+1)
    return 1./(1./2./(2.*alpha)**n * scipy.special.gamma(n))**0.5

# solve equations
def getMpql(l1, m1, l2, m2, alpha1, alpha2, alpha3):
    '''
    Solve eqn Ax = b, where x = Mpql, for l=0, x^2, y^2, z^2
    '''
    # the matrix A
    N0, N2 = basisnorm(alpha3, 0), basisnorm(alpha3, 2)
    I0  = I(alpha3, 0, 0, 0)
    I2  = I(alpha3, 2, 0, 0)
    I4  = I(alpha3, 4, 0, 0)
    I22 = I(alpha3, 2, 2, 0)

    A0 = N0 * I2 / np.sqrt(4*np.pi)
    A1 = N2 * I4
    A2 = N2 * I22
    A3 = N0 * I0 / np.sqrt(4*np.pi)
    A4 = N2 * I2

    A = np.array([
        [A0, A1, A2, A2],
        [A0, A2, A1, A2],
        [A0, A2, A2, A1],
        [A3, A4, A4, A4]
    ])

    # the vector b
    NP = basisnorm(alpha1, l1)
    NQ = basisnorm(alpha2, l2)

    gamma0 = gammaBar(0, l1, l2, alpha1, alpha2)
    gamma2 = gammaBar(2, l1, l2, alpha1, alpha2)

    N00 = np.sqrt(4*np.pi)
    N20 = 4*np.sqrt(np.pi/5)
    N22 = 4*np.sqrt(np.pi/15)

    L = lambda l, m: l**2 + l + m
    NC00 = N00*CG[L(l1,m1), L(l2,m2), L(0,0)]
    NC20 = N20*CG[L(l1,m1), L(l2,m2), L(2,0)]
    NC22 = N22*CG[L(l1,m1), L(l2,m2), L(2,2)]

    b = np.array([
        (1/6)*NP*NQ*gamma2*(2 + 3*NC22 - NC20),
        (1/6)*NP*NQ*gamma2*(2 - 3*NC22 - NC20),
        (1/3)*NP*NQ*gamma2*(1 + NC20),
        NP*NQ*gamma0*NC00
    ])

    return np.linalg.solve(A, b)
    

## **Test with simple example**
Note that, since we write down the compensating charge differently, $M$'s are not identical to $C$'s. Specifically, they are related by the following:
$$
C_0 = M_{S_{00}} N_{S_{00}} S_{00}\\
C_2 = M_{x^2} N_{x^2}
$$

In [183]:
M = getMpql(0, 0, 0, 0, alpha1/2, alpha1/2, 4)
print(C0, M[0] * basisnorm(4, 0) / np.sqrt(4*np.pi))
print(C2, M[1] * basisnorm(4, 2))

-2.633944457835777 2.633944457835775
3.192659948891851 -3.1926599488918486


# **Getting the integrals**
Now I am having trouble getting $(PQ|L)$ where PQ's are any two (spherical) AOs and L are the cartesian compensating charge. Specifically, the objective is to get $(PQ|S_{00}), (PQ|x^2), (PQ|y^2), (PQ|z^2)$ from $(PQ|S_{00}), (PQ|r^2), (PQ|S_{20}), (PQ|S_{22})$ through some linear combination. Specifically
$$
x^2 = \frac{12\sqrt{\pi/15} S_{22} - 4\sqrt{\pi/5}S_{20} + 2r^2}{6}\\
y^2 = \frac{-12\sqrt{\pi/15} S_{22} - 4\sqrt{\pi/5}S_{20} + 2r^2}{6}\\
z^2 = \frac{4\sqrt{\pi/5} S_{20} + r^2}{3}
$$
Note that all integrals are periodic integrals evaluated by throwing $\mathbf{G} = 0$ term away. The tricky one to get is $(PQ|r^2)$, it is related with $(PQ|S_{00})$ by the following:
$$
(PQ|S_{00}) = \sum_{\mathbf{G} \neq 0} \frac{4\pi}{G^2 \Omega} \hat{\phi}_{PQ}(\mathbf{G}) \hat{\phi}_{S_{00}} (-\mathbf{G})
$$
whereas
$$
\begin{align}
N_{S_{00}} S_{00} (PQ|r^2) &= \sum_{\mathbf{G} \neq 0} \left(\frac{3}{2\alpha} - \frac{G^2}{4\alpha^2} \right) \frac{4\pi}{G^2 \Omega} \hat{\phi}_{PQ}(\mathbf{G}) \hat{\phi}_{S_{00}} (-\mathbf{G})\\
&= \frac{3}{2\alpha} (PQ|S_{00}) + \frac{\pi}{\Omega \alpha^2} \left( \hat{\phi}_{PQ}(\mathbf{0}) \hat{\phi}_{S_{00}} (\mathbf{0}) - \sum_{\mathbf{G}}  \hat{\phi}_{PQ}(\mathbf{G}) \hat{\phi}_{S_{00}} (-\mathbf{G}) \right)\\
&= \frac{3}{2\alpha} (PQ|S_{00}) + \frac{\pi}{\Omega \alpha^2} N_{S_{00}} N_P N_Q \left( \frac{1}{\sqrt{4\pi}} \left(\frac{\pi}{\alpha}\right) ^{3/2}  \bar{\Gamma} ( l^{(0)}_{PQ}, \alpha_{PQ}) \delta_{L_P = L_Q} - \Omega \mathcal{C}_{L_P L_Q} ^{00} \bar{\Gamma} ( l^{(0)}_{PQ}, \alpha_{PQ} + \alpha)\right)
\end{align}
$$

## **Test correctness of above**

In [19]:
import numpy
from DF import PAWDF

def getVPQL(pmol, gmol):
    mydf = PAWDF(pmol)
    mydf.auxbasis = gmol.basis
    mydf.build()
    dfbuilder = pyscf.pbc.df.rsdf_builder._RSGDFBuilder(pmol, gmol).build()

    # (g|g')
    j2c = dfbuilder.get_2c2e(numpy.zeros((1, 3)))[0]
    j2c_cd, j2c_negative, j2ctag = dfbuilder.decompose_j2c(j2c)
    assert(j2c_negative is None)
    assert(j2ctag == 'CD')

    # (PQ|g)
    # TODO: gamma point only
    eri_3d = numpy.vstack([Lpq[0].copy() for Lpq in mydf.sr_loop(compact=False)])
    eri_3d = numpy.einsum('LM, Mp -> pL', j2c_cd, eri_3d)
    eri_3d = eri_3d.reshape((pmol.nao, pmol.nao, gmol.nao))

    return eri_3d


pbasis = {'He': [[0, [alpha1, 1.]], [1, [alpha1, 1.]]]}

pmol_sph = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = pbasis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L,
    cart = False
)

VPQL_sph = getVPQL(pmol_sph, smol)
print(VPQL_sph.shape)

pmol_cart = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = pbasis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L,
    cart = True
)

VPQL_cart = getVPQL(pmol_cart, cmol)
print(VPQL_cart.shape)

(4, 4, 6)
(4, 4, 7)


In [20]:
print(pmol_cart.ao_labels())
print(pmol_sph.ao_labels())
print(cmol.ao_labels())
print(smol.ao_labels())

['0 He 1s    ', '0 He 2px   ', '0 He 2py   ', '0 He 2pz   ']
['0 He 1s    ', '0 He 2px   ', '0 He 2py   ', '0 He 2pz   ']
['0 He 1s    ', '0 He 3dxx  ', '0 He 3dxy  ', '0 He 3dxz  ', '0 He 3dyy  ', '0 He 3dyz  ', '0 He 3dzz  ']
['0 He 1s    ', '0 He 3dxy  ', '0 He 3dyz  ', '0 He 3dz^2 ', '0 He 3dxz  ', '0 He 3dx2-y2']


Check correctness by calculating $(PQ | r^2)$ in 2 ways. First using
$$
N_{S_{20}} r^2 = 3z^2 - 4\sqrt{\pi/5} S_{20}
$$

In [107]:
VPQrr1 = VPQL_cart[:, :, -1]/basisnorm(alpha2, 2) * 3 - 4*np.sqrt(np.pi/5) * VPQL_sph[:, :, 3]/basisnorm(alpha2, 2)
VPQrr1

array([[ 3.48369141e-01, -1.46230662e-33, -1.20358572e-34,
         2.63390297e-34],
       [-1.46230662e-33,  3.42502407e-01,  1.89898345e-19,
         7.94518823e-22],
       [-1.20358572e-34,  1.89898345e-19,  3.42502407e-01,
        -6.90680869e-21],
       [ 2.63390297e-34,  7.94518823e-22, -6.90680869e-21,
         3.42502407e-01]])

Then using
$$
\frac{3}{2\alpha} (PQ|S_{00}) + \frac{\pi}{\Omega \alpha^2} N_{S_{00}} N_P N_Q \left( \frac{1}{\sqrt{4\pi}} \left(\frac{\pi}{\alpha}\right) ^{3/2}  \bar{\Gamma} ( l^{(0)}_{PQ}, \alpha_{PQ}) \delta_{L_P = L_Q} - \Omega \mathcal{C}_{L_P L_Q} ^{00} \bar{\Gamma} ( l^{(0)}_{PQ}, \alpha_{PQ} + \alpha)\right)
$$

In [ ]:
from pyscf.paw.PAWutils import getAlphaAtomsL

alphaP, atomsP, LP, MP = getAlphaAtomsL(pmol_sph._bas, pmol_sph._env, cart=False)

N0 = basisnorm(alpha2, 0)


NP = basisnorm(alphaP, LP)
NPQ = numpy.einsum('P,Q->PQ', NP, NP)

GPQ1 = np.zeros((LP.shape[0], LP.shape[0]))
for P in range(LP.shape[0]):
    for Q in range(LP.shape[0]):
        GPQ1[P, Q] = gammaBar(0, LP[P], LP[Q], alphaP[P], alphaP[Q])
        
GPQ2 = np.zeros((LP.shape[0], LP.shape[0]))
for P in range(LP.shape[0]):
    for Q in range(LP.shape[0]):
        GPQ2[P, Q] = gammaBar(0, LP[P], LP[Q], alphaP[P], alphaP[Q]+alpha2)

delta = (
    (LP[:, None] == LP[None, :]) &
    (MP[:, None] == MP[None, :])
).astype(int)

CGPQ = np.zeros((LP.shape[0], LP.shape[0]))
for P in range((LP.shape[0])):
    for Q in range((LP.shape[0])):
        llP = LP[P]**2 + LP[P] + MP[P]
        llQ = LP[Q]**2 + LP[Q] + MP[Q]
        CGPQ[P, Q] = CG[llP, llQ, 0]

VPQrr2 = 3/2/alpha2 * VPQL_sph[:, : , 0] +\
         np.pi/mol.vol/alpha2**2 * N0 * NPQ *\
         (1/np.sqrt(4*np.pi) * (np.pi/alpha2)**(3/2) * GPQ1 * delta -\
          mol.vol * CGPQ * GPQ2)


VPQrr2 = VPQrr2/basisnorm(alpha2, 0)*np.sqrt(4*np.pi)


The two ways give the same result!

In [130]:
print(np.max((np.abs(VPQrr1-VPQrr2))))

3.3375807673152735e-10


This is a nice result, but it turns out there is a simpler way to get $(PQ|x^2)$ etc. where PQ are spherical GTOs from the integrals where all GTOs are cartesian, because there is a 1-to-1 mapping from cartesian GTOs to spherical GTOs...

In [146]:
# let us use a bigger primitive basis to illustrate how this works in general
pbasis = {'He': [[0, [alpha1, 1.]], [1, [alpha1, 1.]], [2, [alpha1, 1.]]]}

pmol_sph = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = pbasis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L,
    cart = False
)

VPQL_sph = getVPQL(pmol_sph, smol)
print(VPQL_sph.shape)

pmol_cart = pgto.M(
    atom = f'He {x} {x} {x}',
    basis = pbasis,
    ke_cutoff=ke_cutoff,
    a = np.eye(3)*L,
    cart = True
)

VPQL_cart = getVPQL(pmol_cart, cmol)
print(VPQL_cart.shape)

(20, 18)


ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (6,6)->(6,6) (12,324)->(324,newaxis,12) 

In [142]:
print(np.max(np.abs(pmol_cart.cart2sph_coeff()-pmol_sph.cart2sph_coeff())))
print(pmol_cart.cart2sph_coeff().shape)
print(smol.ao_labels())
print(cmol.ao_labels())

0.0
(10, 9)
['0 He 1s    ', '0 He 3dxy  ', '0 He 3dyz  ', '0 He 3dz^2 ', '0 He 3dxz  ', '0 He 3dx2-y2']
['0 He 1s    ', '0 He 3dxx  ', '0 He 3dxy  ', '0 He 3dxz  ', '0 He 3dyy  ', '0 He 3dyz  ', '0 He 3dzz  ']


In [144]:
car2sph = pmol_cart.cart2sph_coeff()
gcar2sph = cmol.cart2sph_coeff()
VPQL_cart2sph = np.einsum('PQL, Pp, Qq, Ll -> pql', VPQL_cart, car2sph, car2sph, gcar2sph)
np.max(np.abs(VPQL_cart2sph-VPQL_sph))

np.float64(1.2573275753879898e-14)

# Scratch Work

In [14]:
cOnR_0.sum() * dv

np.float64(0.6960409995630206)

In [ ]:
(np.pi/alpha2) ** (3/2)

0.6960409996039635

In [ ]:
print(cmol.ao_labels())
print(smol.ao_labels())

['0 He 1s    ', '0 He 3dxx  ', '0 He 3dxy  ', '0 He 3dxz  ', '0 He 3dyy  ', '0 He 3dyz  ', '0 He 3dzz  ']
['0 He 1s    ', '0 He 3dxy  ', '0 He 3dyz  ', '0 He 3dz^2 ', '0 He 3dxz  ', '0 He 3dx2-y2']


In [17]:
s0 = cmol.intor('int1e_ovlp_sph')

In [18]:
c = cmol.cart2sph_coeff()
s1 = c.T.dot(cmol.intor('int1e_ovlp_cart')).dot(c)

In [19]:
print(abs(s1-s0).sum())

4.9926772078939e-16


In [20]:
c[:, 0]

array([1., 0., 0., 0., 0., 0., 0.])

In [21]:
0.5*np.sqrt(5/np.pi)

np.float64(0.6307831305050401)

In [22]:
zz = cOnR[:, -1] * ((225*np.pi/(2 * 128**2 * (alpha2)**7))**(1/4))
l=2
n=l+1.5
zzg = cOnR[:, -1] * (gamma(n)/(2 * (2*alpha2)**n))**(1/2)
print(zz.sum() * dv)
print(zzg.sum() * dv)

0.0870051249501553
0.0870051249501553


In [23]:
print(((225*np.pi/(2 * 128**2 * alpha2**5))**(1/2)))
print(gamma(n)/(2 * (2*alpha2)**n)*4)

0.004589773452083131
0.0045897734520831315


In [24]:
(0.5 * (np.pi**3/alpha2**5)**0.5)

0.08700512495049544

In [25]:
gamma(3.5)

np.float64(3.323350970447843)

In [26]:
15/8 * np.sqrt(np.pi)

np.float64(3.323350970447842)

In [ ]:
aoOnR_pmol_sph = pmol_sph.pbc_eval_gto('GTOval', Rgrid)
aoOnR_pmol_cart = pmol_cart.pbc_eval_gto('GTOval', Rgrid)
aoOnR_smol = smol.pbc_eval_gto('GTOval', Rgrid)
aoOnR_cmol = cmol.pbc_eval_gto('GTOval', Rgrid)

print(aoOnR_pmol_sph[:, 0].sum() - aoOnR_pmol_cart[:, 0].sum())
print((aoOnR_pmol_sph[:, 1]**2).sum() - (aoOnR_pmol_cart[:, 1]**2).sum())
print((aoOnR_pmol_sph[:, 2]**2).sum() - (aoOnR_pmol_cart[:, 2]**2).sum())
print((aoOnR_pmol_sph[:, 3]**2).sum() - (aoOnR_pmol_cart[:, 3]**2).sum())
print((aoOnR_pmol_sph[:, 1]**2).sum())

print(aoOnR_cmol[:, 0].sum() - aoOnR_smol[:, 0].sum())
print((aoOnR_cmol[:, 2]**2).sum())
print((aoOnR_cmol[:, 2]**2 * 15/4/np.pi).sum() - (aoOnR_smol[:, 1]**2).sum())

0.0
0.0
0.0
0.0
275.7520414648289
0.0
231.0134884095744
0.0


In [ ]:
smol.cart2sph_coeff(normalized='all')

array([[ 0.28209479,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        , -0.31539157,  0.        ,
         0.54627422],
       [ 0.        ,  1.09254843,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  1.09254843,
         0.        ],
       [ 0.        ,  0.        ,  0.        , -0.31539157,  0.        ,
        -0.54627422],
       [ 0.        ,  0.        ,  1.09254843,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.63078313,  0.        ,
         0.        ]])

In [ ]:
print(np.max(np.abs(VPQL_cart[:, :, 0] - VPQL_sph[:, :, 0])))
print(np.max(np.abs(VPQL_cart[:, :, 2] - VPQL_sph[:, :, 1]))) ## this is so weird!!!
print(np.max(np.abs(VPQL_cart[:, :, 3] - VPQL_sph[:, :, -2]))) ## this is so weird!!!
print(np.max(np.abs(VPQL_cart[:, :, -2] - VPQL_sph[:, :, 2]))) ## this is so weird!!!
print(np.max(np.abs(VPQL_cart[:, :, -1] - VPQL_sph[:, :, 3]))) ## this is so weird!!!
print(VPQL_cart[:, :, -3])

# the L functions are normalized against \int (r^l e^{-ar^2} r^2 dr

# aos
pmol_s00_cart   = aoOnR_pmol_cart[:, 0]
cmol_xx_cart    = aoOnR_cmol[:, 1]
cmol_s00_cart   = aoOnR_cmol[:, 0]
# cmol_s00_cart   = aoOnR_cmol[:, 0] / aoOnR_cmol[:, 0].sum()/dv

2.042810365310288e-14
0.014436934221433273
0.0144369342214333
0.0144369342214333
3.4280945621352874
[[ 3.42809456e+00 -7.70371978e-33  2.64815367e-33 -1.30963236e-32]
 [-7.70371978e-33  3.26636775e+00  9.70082594e-19 -4.80516246e-20]
 [ 2.64815367e-33  9.70082594e-19  3.57835502e+00 -8.36974431e-20]
 [-1.30963236e-32 -4.80516246e-20 -8.36974431e-20  3.26636775e+00]]


In [ ]:
(cmol_s00_cart**2).sum() * dv
(cmol_s00_cart).sum() * dv

np.float64(1.4031041454516824)

In [ ]:
# calculate (0 0 | 0) by hand
N = gammaBar(0, 0, 0, alpha2, alpha2) / gammaBar(0, 0, 0, alpha2, 0)
print(VPQL_cart[0, 0, 0])
print(VPQL_sph[0, 0, 0]*N*np.pi*4)
v_s00_cart = np.fft.ifftn((np.fft.fftn((pmol_s00_cart**2).reshape(mesh)) * FF)).flatten()
print(np.dot(v_s00_cart, cmol_s00_cart)*dv.real)

# calculate (0 0 | x^2) by hand
print(VPQL_cart[:, :, 1])
v_s00_cart = np.fft.ifftn((np.fft.fftn((pmol_s00_cart**2).reshape(mesh)) * FF)).flatten()
print(np.dot(v_s00_cart, cmol_xx_cart)*dv)

2.653710022293233
11.79012298086637
(2.6537100088532877-1.3475176457868692e-16j)
[[ 3.42809456e+00  2.04148574e-32  8.37779526e-33 -2.23407874e-32]
 [ 2.04148574e-32  3.57835502e+00 -1.20617492e-18  5.38395317e-20]
 [ 8.37779526e-33 -1.20617492e-18  3.26636775e+00  2.96461532e-20]
 [-2.23407874e-32  5.38395317e-20  2.96461532e-20  3.26636775e+00]]
(3.428094585341556-1.8446597254661256e-16j)


In [38]:
VPQL_cart2sph = numpy.einsum("PQL, LG -> PQG", VPQL_cart, smol.cart2sph_coeff())
np.max(np.abs(VPQL_sph - VPQL_cart2sph))

np.float64(2.042810365310288e-14)

In [118]:
smol.cart2sph_coeff()[:, 3]

array([ 0.        , -0.31539157,  0.        ,  0.        , -0.31539157,
        0.        ,  0.63078313])

In [40]:
VPQL_cart2sph[:, :, 3]

array([[ 0.00000000e+00, -1.37277385e-32, -4.54807267e-33,
         1.37884808e-32],
       [-1.37277385e-32, -9.83981538e-02,  1.11515975e-18,
         3.74227698e-21],
       [-4.54807267e-33,  1.11515975e-18, -9.83981538e-02,
        -2.58620682e-20],
       [ 1.37884808e-32,  3.74227698e-21, -2.58620682e-20,
         1.96796308e-01]])

In [41]:
VPQL_sph[:, :, 3]

array([[ 1.08654115e-17, -9.62964972e-34, -4.85244068e-34,
         1.46551232e-33],
       [-9.62964972e-34, -9.83981538e-02, -2.07047794e-19,
         9.54038736e-22],
       [-4.85244068e-34, -2.07047794e-19, -9.83981538e-02,
        -5.65473705e-23],
       [ 1.46551232e-33,  9.54038736e-22, -5.65473705e-23,
         1.96796308e-01]])